# Per-photon hit map

Requires a per-photon run (``--output-mode per_photon``):

1. Detection-fraction distribution over cases (linear + log y).
2. Hit-time distribution (linear + log y — scatter tail).
3. 3D scatter of sensors coloured by total hits over the run (linear + log).
4. 1D distribution of hits-per-sensor.


In [ ]:
import sys
sys.path.append('../../../../')  # notebooks/ → photon_shotgun/ → production/ → lucid/ → repo root

import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from lucid.production.photon_shotgun.io import (
    load_shotgun_waveform, load_shotgun_per_photon,
)
from lucid.production.photon_shotgun.viz import (
    scatter_sensors_3d, plot_hist_lin_log,
)
from lucid.geometry import generate_detector

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)


def _resolve_geom(meta):
    geom_path = meta.get('detector_config', '')
    if isinstance(geom_path, bytes):
        geom_path = geom_path.decode()
    for candidate in ['../../../../' + geom_path, geom_path]:
        if os.path.exists(candidate):
            return candidate
    return geom_path


In [ ]:
PP_PATH = '../../../../runs/shotgun_SK_1k_per_photon.h5'

out = load_shotgun_per_photon(PP_PATH)
meta = dict(out['meta'])
print('n_cases, n_photons:', int(meta['n_cases']), int(meta['n_photons']))

detected = out['detected']
sensor_id = out['sensor_id']
hit_time = out['hit_time']

frac_per_case = detected.mean(axis=1)
print(f"detection fraction: mean={frac_per_case.mean():.4f}  "
      f"std={frac_per_case.std():.4f}  "
      f"[{frac_per_case.min():.4f}, {frac_per_case.max():.4f}]")


## Detection-fraction histogram (per case)


In [ ]:
plot_hist_lin_log(frac_per_case, bins=60,
                  title='detection fraction per case',
                  xlabel='fraction', color='steelblue')
plt.show()


## Detected-photon hit times


In [ ]:
det_times = hit_time[detected]
plot_hist_lin_log(det_times, bins=200,
                  title='hit times (all detected photons)',
                  xlabel='time (ns)', color='coral')
plt.show()


## Per-sensor hit-count 3D scatter — linear + log


In [ ]:
det = generate_detector(_resolve_geom(meta))
sensor_points = np.asarray(det.all_points)
num_sensors = sensor_points.shape[0]

valid = detected & (sensor_id >= 0)
hits_per_sensor = np.bincount(sensor_id[valid], minlength=num_sensors).astype(np.float32)
print(f"{(hits_per_sensor > 0).sum()} / {num_sensors} sensors had ≥1 hit")

fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(121, projection='3d')
scatter_sensors_3d(sensor_points, hits_per_sensor, ax=ax1, log=False,
                   title='hits per sensor (linear)', cbar_label='hits')
ax2 = fig.add_subplot(122, projection='3d')
scatter_sensors_3d(sensor_points, hits_per_sensor, ax=ax2, log=True,
                   title='hits per sensor (log)', cbar_label='hits')
plt.show()


## Per-sensor hit-count distribution


In [ ]:
plot_hist_lin_log(hits_per_sensor[hits_per_sensor > 0], bins=80,
                  title='per-sensor hit count (sensors with ≥1 hit)',
                  xlabel='hits', color='seagreen')
plt.show()
